# 3장 07. Argo CD와 KServe manifest 확인

이 notebook은 Kubernetes에 MLflow tracking service를 먼저 배포한 뒤, KServe `InferenceService`로 모델 serving 후보를 배포하는 manifest 흐름을 확인합니다. 실제 sync는 Argo CD가 Git의 원하는 상태를 cluster live state로 맞추는 단계입니다.


## 기본 개념: GitOps, Argo CD, KServe

Kubernetes 기본 개념은 앞의 `05_kubernetes_concepts.ipynb`에서 이미 확인했습니다. 여기서는 배포 파일을 읽는 데 필요한 세 가지만 봅니다.

GitOps는 Git에 적힌 배포 파일을 원하는 상태로 보고, cluster가 그 상태와 같아지도록 맞추는 방식입니다. Argo CD는 Git repository의 특정 path를 보고 Kubernetes cluster에 적용합니다. 그래서 강의에서는 `kubectl apply`를 직접 반복하기보다, Argo CD `Application`이 어떤 Git path를 바라보는지 확인합니다.

KServe는 Kubernetes 위에서 모델 serving endpoint를 선언하는 도구입니다. `InferenceService`에는 어떤 모델을 어떤 storage URI에서 읽을지, 어떤 predictor로 띄울지 같은 정보가 들어갑니다.

이번 배포 순서는 MLflow tracking 관련 Kubernetes resource를 먼저 확인하고, 그다음 KServe `InferenceService`를 확인하는 흐름입니다. live cluster가 없으면 sync 성공이라고 쓰지 않고, manifest inspection까지만 확인했다고 기록합니다.


In [ ]:
import utils as runtime

prepared = await runtime.prepare_notebook()
pd = prepared.pandas
aiq = prepared.aiq_lite


In [ ]:
from pathlib import Path

application_candidates = [
    Path("demos/ch03_docker_kubernetes/argocd/application.yaml"),
    Path("../../demos/ch03_docker_kubernetes/argocd/application.yaml"),
    Path("artifacts/deployment/chapter_03/argocd/application.yaml"),
    Path("../artifacts/deployment/chapter_03/argocd/application.yaml"),
]
inference_candidates = [
    Path("demos/ch03_docker_kubernetes/gitops/base/inferenceservice.yaml"),
    Path("../../demos/ch03_docker_kubernetes/gitops/base/inferenceservice.yaml"),
    Path("artifacts/deployment/chapter_03/gitops/base/inferenceservice.yaml"),
    Path("../artifacts/deployment/chapter_03/gitops/base/inferenceservice.yaml"),
]

application_path = next(path for path in application_candidates if path.exists())
inference_path = next(path for path in inference_candidates if path.exists())
application = application_path.read_text()
inference = inference_path.read_text()

print("application:", application_path)
print("inferenceservice:", inference_path)


## 1. Argo CD Application에서 Git path 확인

Argo CD는 어떤 Git 경로를 클러스터 상태로 맞출지 선언합니다. 이 경로가 잘못되면 MLflow나 KServe manifest가 있어도 cluster에 적용되지 않습니다.


In [ ]:
for keyword in ["repoURL", "targetRevision", "path", "namespace"]:
    print(keyword, "=>", keyword in application)


## 2. KServe InferenceService에서 모델 후보 확인

KServe manifest에는 serving resource와 model artifact 위치가 들어갑니다. 여기서는 `storageUri`가 앞에서 확인한 MLflow/model artifact 흐름과 이어지는지 봅니다.


In [ ]:
for keyword in ["InferenceService", "risk-classifier", "candidate", "storageUri"]:
    print(keyword, "=>", keyword in inference)
